# Ticket agent with Agent Service Toolkit

Process the **first 15 tickets**, TKT-100000 through TKT-100014, with a real OpenAI model served by [JoshuaC215/agent-service-toolkit](https://github.com/JoshuaC215/agent-service-toolkit). Microsoft's Agent Governance Toolkit checks the four CSV policies. Expected result: **14 response drafts and one human-review item**.

This is a separate demo: its conda environment, service on port 8081, credentials, checkpoints, SQLite decisions, and audit logs belong to this directory. The Langflow demo remains separate.

Start the service using the setup instructions. Read the tickets and policies, inspect the first ticket's context, run the batch, and review its single escalation. Rerun the batch to show saved state. Inspect category memory and verify the signed audit. Appendix A connects this design to context engineering.

Ticket actions are local simulations. The model writes drafts and an internal review note; no customer message is sent and no account is changed. Redacted ticket text is sent to OpenAI.

## 1. Set up conda and the toolkit service

Run these commands in a terminal **inside `agent_service_demo`**. The toolkit requires Python 3.12 or newer, so use a separate environment from the Langflow demo.

```bash
conda create -n agent-service-tickets python=3.12 pip -y
conda activate agent-service-tickets
git clone https://github.com/JoshuaC215/agent-service-toolkit.git toolkit
git -C toolkit checkout fe3b2dcc84d41e6f7f345fd5830f2e12a978c10a
python -m pip install ./toolkit -r requirements-demo.txt
python -m ipykernel install --user --name agent-service-tickets --display-name "Python (agent-service-tickets)"
```

For an existing clone, keep it and run the checkout/install steps. The revision is also recorded in `TOOLKIT_REVISION`; the launcher checks it. The toolkit installs its full dependencies, which can take several minutes. This setup uses conda and pip; Docker and uv are unnecessary.

A local `.env` has been created from the original demo's OpenAI settings, with a new service token. Its permissions allow only its owner to read it. Keep it untracked. If starting from a copy without `.env`, copy `.env.example` and replace the placeholders:

```dotenv
OPENAI_API_KEY=your-openai-key
OPENAI_MODEL=gpt-4.1-mini
AUTH_SECRET=your-long-random-service-token
PORT=8081
```

`AUTH_SECRET` authenticates notebook requests to this service. `OPENAI_API_KEY` authenticates model calls. No Langflow key is needed.

Start the service in this terminal and leave it running:

```bash
conda activate agent-service-tickets
python run_service.py
```

`Application startup complete` means the service has started. With the default port, open http://127.0.0.1:8081/health to check for `{"status":"ok"}`. This service has no homepage: opening http://127.0.0.1:8081/ returns `404 Not Found`, as do automatic browser requests for `/favicon.ico` and Apple touch icons. These responses do not indicate a startup failure. The notebook is the demo interface.

The launcher uses the toolkit's unmodified FastAPI service, SQLite checkpointer, and our small ticket-agent registry. It sets the built-in `openai-compatible` model alias to the actual `OPENAI_MODEL` at OpenAI's official endpoint. This preserves `gpt-4.1-mini` without editing the toolkit's model enum. Model access depends on your OpenAI account. Optional LangSmith/Langfuse tracing is disabled; local governance auditing remains enabled.

In a second terminal, also inside this directory:

```bash
conda activate agent-service-tickets
python -m jupyter lab ticket_agent_demo.ipynb
```

Select **Python (agent-service-tickets)**. API documentation is at http://127.0.0.1:8081/docs. Restart `python run_service.py` after changing `.env`, the agent code, or its instructions. Stop the service with Ctrl+C; files remain on disk.

**Prepare the run.** The notebook selects the local ticket service and the model that will write the responses.

In [4]:
from collections import Counter
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import JSON, display
from demo_support import TicketDemo

ROOT = Path.cwd()
load_dotenv(ROOT / '.env', override=True)
MODEL = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')
SERVICE_URL = f"http://127.0.0.1:{os.getenv('PORT', '8081')}"
OUTPUT = ROOT / 'demo_output'
print(f'Service: {SERVICE_URL}; OpenAI model: {MODEL}')


Service: http://127.0.0.1:8081; OpenAI model: gpt-4.1-mini


## 2. Load 15 tickets and four policies

Only TKT-100000 receives synthetic lifecycle facts: it is awaiting a customer reply and has been inactive for seven business days. POL-003 allows closure after five days; POL-004 forbids General Inquiry closure before ten. The toolkit's `deny_overrides` strategy blocks closure and selects escalation. The other 14 tickets request ordinary drafts under POL-001. POL-002 permits recording an escalation. Unlisted actions are denied.

The original CSV is copied intact; only its first 15 rows are read. Historical resolution times and satisfaction scores are excluded from model context. The helper module keeps CSV parsing, HTTP, and persistence code out of the demo cells.

**Load tickets and rules.** It reads the first 15 tickets and four policies. These policies allow drafting and human escalation, and set limits on when tickets may be closed.

In [5]:
if 'demo' in globals():
    demo.close()
demo = TicketDemo(ROOT, MODEL, SERVICE_URL, os.environ['AUTH_SECRET'], output=OUTPUT)
tickets, policies = demo.tickets, demo.policies
# Expected: 15 tickets, TKT-100000 through TKT-100014, and four policies.
print(f'{len(tickets)} tickets: {tickets[0]["Ticket_ID"]} through {tickets[-1]["Ticket_ID"]}')
for policy in policies:
    print(f'{policy["policy_id"]}: {policy["policy_text"]}')


15 tickets: TKT-100000 through TKT-100014
POL-001: Drafting a support response is allowed; sending messages or changing accounts is outside this demo.
POL-002: Recording a human escalation is allowed.
POL-003: Tickets awaiting a customer response may be closed after 5 business days of inactivity.
POL-004: General Inquiry tickets must remain open for at least 10 business days before automatic closure.


## 3. Inspect context, governance, state, and memory

| Concept | What this demo shows |
| --- | --- |
| Context | One redacted ticket, four policies, trusted lifecycle facts, governance feedback, and category memory. |
| State | Saved decisions in `demo_output/state.sqlite3`; the service also persists LangGraph thread checkpoints in a separate SQLite file. |
| Memory | Counts of previous outcomes for this issue category, retrieved from the saved decisions and placed in context. |
| Audit | Toolkit hash-chain records and HMAC signatures, with policy, agent, context, and proposal fingerprints. |

The host checks the requested action before the LLM sees it and checks the returned action before saving a simulated result. Memory cannot override policy. The model receives instructions as a system message and the context as a separate user message. Each ticket uses a separate service thread. The agent consumes the latest bounded context; it does not replay an accumulating conversation into every prompt.

Inspect the first ticket. Both closure policies should match and `allowed_next_action` should be `escalate`. These local checks do not test the provider connection.

**Build the context.** Each ticket is accompanied by its redacted details, relevant facts, policies, and a summary of earlier outcomes in its category. The first ticket's preview shows empty memory because there are no earlier saved outcomes in that category yet.

**Check what is allowed.** The first ticket, TKT-100000, is given a seven-day inactivity scenario. One policy allows closure after five business days; another requires General Inquiry tickets to remain open for ten. The stricter restriction blocks closure and sends the ticket toward human review.

In [6]:
# Expected: POL-003 and POL-004 match; conflict=True; allowed_next_action='escalate'.
display(JSON(demo.context(tickets[0]), expanded=True))


<IPython.core.display.JSON object>

## 4. Run the real LLM and review the escalation

First display the agents and model aliases advertised by the toolkit service. Each uncached ticket then makes one HTTP request to `/ticket-agent/invoke`. Inside the service, the LangGraph agent uses OpenAI structured output for `action`, `draft`, and `reason`. The host validates that proposal and checks policy again.

Fourteen requests should produce customer response drafts. The conflicting closure request should produce an internal Support/Operations review note naming POL-003 and POL-004. Escalation is saved to a local queue file; it does not notify another person or approve closure.

Errors stop processing and are audited. HTTP errors and model failures show redacted diagnostics; no scripted planner is substituted. For connection errors, check the terminal running `python run_service.py`. For authentication errors, ensure both processes read this directory's `.env`. For provider-key, quota, or model-access errors, update the key or model and restart the service. Successful decisions are reused on rerun.

**Confirm the service is ready.** The notebook checks that the ticket agent and a model are available before processing the tickets.

In [7]:
info = demo.request('GET', '/info')
# Expected: agents includes 'ticket-agent'; models includes 'openai-compatible'.
print('Agents:', [agent['key'] for agent in info['agents']])
print('Models:', info['models'])


Agents: ['ticket-agent']
Models: ['gpt-5-mini', 'gpt-5-nano', 'gpt-5.1', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'openai-compatible']


**Ask the model for an allowed proposal.** The model writes a response draft for each of the other 14 tickets and an internal review note for the blocked ticket. The workflow checks each proposal against the rules again before accepting it.

**Demonstrate reuse on a rerun.** All saved rows show `Cached=False`: these results were newly generated. Rerunning the unchanged batch should show `Cached=True`, reusing saved decisions without asking the model again. Changes to a ticket, policies, or model require a fresh result.

In [8]:
# Expected: TKT-100000 needs_human_review; the other 14 tickets are drafted.
# Rerun this cell: every row should show Cached=True, with no new model requests.
results = []
print(f'{"Ticket":<14} {"Status":<22} {"Action":<18} Cached')
for ticket in tickets:
    result = demo.process(ticket)
    results.append(result)
    print(f'{result["ticket_id"]:<14} {result["status"]:<22} {result["action"]:<18} {result["cached"]}', flush=True)


Ticket         Status                 Action             Cached
TKT-100000     needs_human_review     escalate           False
TKT-100001     drafted                draft_response     False
TKT-100002     drafted                draft_response     False
TKT-100003     drafted                draft_response     False
TKT-100004     drafted                draft_response     False
TKT-100005     drafted                draft_response     False
TKT-100006     drafted                draft_response     False
TKT-100007     drafted                draft_response     False
TKT-100008     drafted                draft_response     False
TKT-100009     drafted                draft_response     False
TKT-100010     drafted                draft_response     False
TKT-100011     drafted                draft_response     False
TKT-100012     drafted                draft_response     False
TKT-100013     drafted                draft_response     False
TKT-100014     drafted                draft_response  

**Review and collect the results.** The saved output confirms **14 drafted responses and one human-review item**. The review note explains the conflicting closure rules. Results and the review queue are saved locally. These are simulated actions: drafts are not sent, and the queue does not notify a person.

In [9]:
review = [r for r in results if r['status'] == 'needs_human_review']
(OUTPUT / 'processed_tickets.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
(OUTPUT / 'human_review.json').write_text(json.dumps(review, indent=2), encoding='utf-8')
# Expected counts: {'needs_human_review': 1, 'drafted': 14}.
print(dict(Counter(r['status'] for r in results)))
# Expected review item: TKT-100000, with an internal note about POL-003 and POL-004.
display(JSON(review, expanded=True))


{'needs_human_review': 1, 'drafted': 14}


<IPython.core.display.JSON object>

## 5. Reuse saved state and inspect memory

Rerun the batch cell above to show `Cached=True` without additional model requests. Its fingerprint covers the ticket, trusted facts, policies, model, agent code, instructions, response schema, and toolkit revision. A changed input triggers a fresh proposal.

Category memory persists across kernel restarts because it comes from the saved decisions. The toolkit's SQLite checkpointer preserves service thread state; its default cross-thread Store is in memory and is not used as our durable category memory. A recorded draft is not evidence of a resolved customer issue.

**Keep state.** The notebook confirms **15 saved decisions**. State records progress and outcomes so the workflow can pick up later. Each ticket also has its own separate conversation history.

**Use memory across tickets.** Memory summarizes previous outcomes for tickets in the same category and adds that summary to later context. It comes from saved decisions, so it survives restarting the notebook. A saved draft counts as a draft, not a resolved issue; memory cannot override policy.

In [10]:
# Expected: outcome counts for this category and 15 saved business decisions.
display(JSON(demo.context(tickets[1])['memory']))
print('Saved decisions:', demo.db.execute('SELECT count(*) FROM decisions').fetchone()[0])
print('Service thread:', results[0]['thread_id'])


<IPython.core.display.JSON object>

Saved decisions: 15
Service thread: 90431301460a4c01acdf598539a1cc63-TKT-100000


## 6. Verify the audit evidence

Signed audit records contain ticket IDs, policy and agent hashes, matched rules, context/proposal hashes, decisions, and service thread/run IDs. Raw ticket text and credentials are omitted. The audit key has owner-only permissions. A checkpoint of the line count and final hash detects truncation; protect it separately for stronger evidence.

The cell displays verification results for the real log and a modified temporary copy. Expected values are comments; the displayed values come from the actual checks. Rerun it after rerunning the batch so its checkpoint includes the new skip events.

**Verify the audit trail.** The saved run contains **45 audit records**. The original record passes verification, while a deliberately altered copy fails. The audit provides evidence of the workflow's actions; it is separate from the memory used to write responses.

In [11]:
# Expected: audit_valid=True and tampered_copy_valid=False.
report = demo.audit_report()
display(JSON(report, expanded=True))
print('Artifacts:', OUTPUT)


<IPython.core.display.JSON object>

Artifacts: /Users/sabersoleymani/workspace/teaching/msa8770_fall2026/agent_service_demo/demo_output


## Limits and references

Inactivity facts are synthetic. Name/email redaction is a small filter rather than complete PII detection. Business-state and audit writes are separate transactions, so a crash can require reconciliation. An operator controlling both the audit key and files can rewrite local evidence. Human review is a local artifact; the workflow does not resume an approved closure.

For a fresh presentation, stop the service and rename this directory's `demo_output` folder before restarting the service and notebook. Keep any audit files, keys, and checkpoints together if retaining old evidence. The Langflow demo's output directory is independent.

- [Agent Service Toolkit](https://github.com/JoshuaC215/agent-service-toolkit), [service API implementation](https://github.com/JoshuaC215/agent-service-toolkit/blob/main/src/service/service.py), and [model integration](https://github.com/JoshuaC215/agent-service-toolkit/blob/main/src/core/llm.py).
- [Microsoft Agent Governance Toolkit](https://github.com/microsoft/agent-governance-toolkit); this demo pins `agent-governance-toolkit-core==4.1.0`.
- [Official OpenAI structured output documentation](https://developers.openai.com/api/docs/guides/structured-outputs) and [GPT-4.1 mini](https://developers.openai.com/api/docs/models/gpt-4.1-mini).

Implementation is in `agents/ticket_agent.py`, `ticket_contract.py`, and `demo_support.py`. Run `python -m unittest -v test_demo test_agent` in the configured conda environment for checks that substitute only HTTP/model responses. Those tests do not claim a live OpenAI run.

## Appendix A. Context engineering

LangChain's [Context Engineering: How Deep Agents Remember](https://www.langchain.com/blog/context-engineering-deep-agents-remember) discusses offloading, summarization, skills, subagents, and long-term memory.

This short demo supplies one redacted ticket and a compact category summary per request. SQLite retains decisions outside the prompt. Toolkit thread checkpoints record conversation state, while separate ticket threads prevent unrelated ticket histories from mixing. The signed audit remains evidence rather than a memory source for the model.

Long investigations could retrieve attachments, summarize threads, and load support playbooks on demand. Specialist subagents could help with larger investigations. Those extensions are outside this 15-ticket demonstration; no Deep Agents dependency is needed.

Memory helps compose responses. Permission still comes from the host's policy checks; remembered outcomes cannot authorize closure or resolve a policy conflict.